<p><font size="6" color='grey'> <b>
KI-Agenten. Verstehen. Anwenden. Gestalten.
</b></font> </br></p>



<p><font size="5" color='grey'> <b>
RAG-Agent
</b></font> </br></p>

---

In [ ]:
#@title 🛠️ Umgebung einrichten{ display-mode: "form" }
!uv pip install --system -q git+https://github.com/ralf-42/Agenten.git#subdirectory=04_modul

import os
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_PROJECT"]    = "M14-RAG-Agent"
os.environ["LANGSMITH_ENDPOINT"]   = "https://eu.api.smith.langchain.com"

from genai_lib.utilities import (
    check_environment,
    get_ipinfo,
    setup_api_keys,
    mprint,
    install_packages,
    mermaid,
    get_model_profile,
    extract_thinking,
    load_prompt,
    show_trace
)

setup_api_keys(['OPENAI_API_KEY', 'LANGSMITH_API_KEY'], create_globals=False)
print()
check_environment()
print()
get_ipinfo()

# Modell-Konfiguration — Rollen als Konstanten
from genai_lib.model_config import BASELINE, ROUTER, JUDGE, PLANNER, WORKER, WORKER_PREMIUM, CODING, EMBEDDINGS

In [ ]:
#@title 📦 Installationen { display-mode: "form" }
install_packages([
                ('markitdown[all]', 'markitdown'),
                'langchain_huggingface',
                ('"unstructured[all-docs]>=0.11.2"', 'unstructured'),
                ])

In [ ]:
#@title 📂 Dokumente { display-mode: "form" }
import shutil
from genai_lib.utilities import copy_from_github

shutil.rmtree("/content/files", ignore_errors=True)
copy_from_github(
    source="ralf-42/Agenten/02_daten/01_text",
    target="/content/files",
    mask="biografien_*",
)

# 1 | Übersicht
---



In *RAG-Chain mit LangChain* haben wir eine **RAG-Chain** gebaut: deterministisch, linearer Ablauf.  
Jetzt machen wir den nächsten Schritt: **RAG als Tool für einen Agenten**.

**Von der Chain zum Agent**

| Merkmal | RAG-Chain (M10) | RAG-Agent (M11) |
|---------|----------------|----------------|
| Ablauf | Immer: Retrieve → Generate | Nur bei Bedarf: Agent entscheidet |
| RAG-Nutzung | Immer aktiv | Tool, das der Agent wählt |
| Weiteres Wissen | Nur Vektordatenbank | Eigenes LLM-Wissen + RAG + andere Tools |
| Flexibilität | Niedrig | Hoch |
| Komplexität | Gering | Mittel |

**Was wir bauen**

Ein Agenten-System mit **RAG als einem von mehreren Tools**:
- `suche_wissensdatenbank` – RAG-Tool für spezifische Kursfragen
- `erklaere_konzept` – Direktes LLM-Wissen für allgemeine Erklärungen
- `berechne_tokens` – Einfaches Berechnungstool

In [ ]:
#@markdown   <p><font size="4" color='green'>  flowchart</font> </br></p>

diagram = '''
flowchart TD
    U["👤 Nutzer-Anfrage"] --> A["🤖 RAG-Agent"]
    A --> D{"Welches Tool?"}

    D -->|"Spezifische\nKursfrage"| RAG["📚 suche_wissensdatenbank()\n(ChromaDB Retrieval)"]
    D -->|"Allgemeines\nKonzept"| LLM["🧠 erklaere_konzept()\n(LLM-Wissen)"]
    D -->|"Token-\nBerechnung"| CALC["🔢 berechne_tokens()\n(Formel)"]

    RAG --> ANS["💬 Antwort"]
    LLM --> ANS
    CALC --> ANS
'''

mermaid(diagram, width=650)

# 2 | RAG als Tool
---



Ein **RAG-Tool** ist eine normale LangChain-Tool-Funktion, die intern ChromaDB abfragt.



<p><font color='black' size="5">
Design-Prinzipien für RAG-Tools
</font></p>


| Prinzip | Beschreibung |
|---------|-------------|
| **Klarer Name** | Name sagt dem Agenten, wann das Tool zu nutzen ist |
| **Präzise Beschreibung** | Docstring erklärt genau, welche Fragen das Tool beantwortet |
| **Fehlerbehandlung** | Gibt lesbaren Text zurück, kein `raise` |
| **Formatierte Ausgabe** | Gibt Quellen + Antwort zurück |

In [ ]:
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_core.tools import tool
from langchain.chat_models import init_chat_model
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain.agents import create_agent

# LLM und Embeddings
llm = init_chat_model(BASELINE, temperature=0.0)
embeddings = OpenAIEmbeddings(model=EMBEDDINGS)

In [ ]:
import warnings
warnings.filterwarnings("ignore", message=".*triton.*")

from langchain_community.document_loaders import (
    UnstructuredMarkdownLoader,
    UnstructuredWordDocumentLoader,
    PyPDFLoader,
    UnstructuredFileLoader,
    DirectoryLoader,
)

# Loader-Konfiguration
loader_mapping = {
    "*.md":   UnstructuredMarkdownLoader,
    "*.docx": UnstructuredWordDocumentLoader,
    "*.pdf":  PyPDFLoader,
    "*.txt":  UnstructuredFileLoader,
}

def load_documents_from_directory(directory_path):
    """Lädt Dokumente aus dem Verzeichnis für alle unterstützten Dateitypen."""
    documents = []
    for file_pattern, loader_cls in loader_mapping.items():
        loader = DirectoryLoader(directory_path, glob=file_pattern, loader_cls=loader_cls)
        documents.extend(loader.load())
    return documents

# Dokumente laden + splitten
dokumente_raw = load_documents_from_directory("/content/files")
splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=30)
chunks = splitter.split_documents(dokumente_raw)

# ChromaDB aufbauen (persistent)
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name="kurs_kb",
    persist_directory="chroma_m14"
)
print(f"Wissensdatenbank bereit: {vectorstore._collection.count()} Chunks")

# 3 | RAG-Agent bauen
---



In [ ]:
@tool
def suche_wissensdatenbank(frage: str) -> str:
    """Durchsucht die Wissensdatenbank nach Informationen zu Personen und deren Biografien.
    Nutze dieses Tool bei Fragen zu spezifischen Personen, deren Berufen, Projekten oder Tätigkeiten.
    """
    try:
        ergebnisse = vectorstore.similarity_search_with_score(frage, k=3)
        if not ergebnisse:
            return "Keine relevanten Informationen in der Wissensdatenbank gefunden."

        antwort_teile = []
        for doc, score in ergebnisse:
            if score < 1.5:  # Nur relevante Ergebnisse (L2-Distanz)
                quelle = doc.metadata.get("source", "unbekannt").split("/")[-1]
                antwort_teile.append(f"[{quelle}] {doc.page_content}")

        if not antwort_teile:
            return "Keine ausreichend relevanten Informationen gefunden."

        return "\n\n".join(antwort_teile)
    except Exception as e:
        return f"Fehler bei der Datenbanksuche: {str(e)}"


@tool
def erklaere_konzept(konzept: str) -> str:
    """Erklaert ein allgemeines KI- oder Informatik-Konzept kurz und verstaendlich.
    Nutze dieses Tool bei allgemeinen Konzeptfragen (z.B. was ist ein Embedding?,
    was ist ein Transformer?) die nicht biografie-spezifisch sind.
    """
    erklaer_llm = init_chat_model(BASELINE, temperature=0.2)
    prompt = f"Erklaere das Konzept '{konzept}' in 2-3 Saetzen fuer einen Einsteiger."
    return erklaer_llm.invoke(prompt).content


@tool
def berechne_tokens(text: str, modell: str = "gpt-4o-mini") -> str:
    """Schaetzt die Anzahl der Tokens fuer einen gegebenen Text.
    Nuetzlich um zu verstehen, wie viel ein Text im Kontext kostet.
    """
    zeichen = len(text)
    tokens_grob = zeichen // 3
    kosten_per_1k = 0.00015  # gpt-4o-mini input
    tokens_per_1k = tokens_grob / 1000
    kosten = tokens_per_1k * kosten_per_1k
    return (
        f"Text: {zeichen} Zeichen\n"
        f"Geschaetzte Tokens: ~{tokens_grob} (Faustregel: 3 Zeichen/Token)\n"
        f"Geschaetzte Kosten ({modell}): ~${kosten:.6f}"
    )


tools = [suche_wissensdatenbank, erklaere_konzept, berechne_tokens]
print(f"Tools definiert: {[t.name for t in tools]}")

In [ ]:
# RAG-Agent erstellen
system_prompt = load_prompt("https://github.com/ralf-42/Agenten/blob/main/05_prompt/m14_rag_agent_system_prompt.md", mode="S")

rag_agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt=system_prompt,
    debug=False
)

print("RAG-Agent erstellt")
print(f"Tools: {[t.name for t in tools]}")


# 4 | Wann RAG, wann eigenes Wissen?
---



Der Agent entscheidet selbst, welches Tool er nutzt – aber wie trifft er diese Entscheidung?

**Entscheidungslogik**

Der Agent wählt das Tool basierend auf **drei Faktoren**:

1. **Tool-Name** – Sprechende Namen helfen: `suche_wissensdatenbank` vs. `erklaere_konzept`
2. **Docstring** – Präzise Beschreibung, wann das Tool einzusetzen ist
3. **Kontext der Frage** – Spezifisch (→ RAG) vs. allgemein (→ eigenes Wissen)

In [ ]:
#@markdown   <p><font size="4" color='green'>  flowchart</font> </br></p>

diagram = '''
flowchart TD
    FRAGE["❓ Nutzerfrage"]

    FRAGE --> SPEZ{{"Kurs-spezifisch?\n(LangChain, LangGraph,\nRAG, ChromaDB)"}}
    SPEZ -->|"Ja"| RAG["📚 suche_wissensdatenbank()\nChromaDB Retrieval"]
    SPEZ -->|"Nein"| ALLG{{"Allgemeines\nKonzept?"}}

    ALLG -->|"Ja"| LLM["🧠 erklaere_konzept()\nLLM-Wissen"]
    ALLG -->|"Nein"| CALC{{"Berechnung\nnoetig?"}}

    CALC -->|"Ja"| TOK["🔢 berechne_tokens()\nFormel"]
    CALC -->|"Nein"| DIRECT["💬 Direkte Antwort"]

    RAG --> ANS["Antwort"]
    LLM --> ANS
    TOK --> ANS
    DIRECT --> ANS
'''

mermaid(diagram, width=650)

In [ ]:
# Test: Agent zeigt unterschiedliche Tool-Wahl
test_fragen = [
    # Erwartet: suche_wissensdatenbank (Biografie-Frage)
    "Welche Methode hat Nadia Khoury entwickelt, um Mondstaub zu nutzen, und für welches Projekt arbeitet sie?",
    # Erwartet: suche_wissensdatenbank (Biografie-Frage)
    "Was hat Dorian Blackwood vor seiner Karriere bei IllusionSec gemacht, und wie hilft er heute Kindern?",
    # Erwartet: erklaere_konzept (allgemeine KI-Frage)
    "Was ist ein Embedding-Vektor in der KI?",
]

run_cfg = {"run_name": "RAG-Agent-Tool-Wahl", "tags": ["rag-agent", "m14"]}

for frage in test_fragen:
    print(f"Frage: {frage}")
    antwort = rag_agent.invoke(
        {"messages": [{"role": "user", "content": frage}]},
        config=run_cfg
    )
    letzte_antwort = antwort["messages"][-1].content
    print(f"Antwort: {letzte_antwort[:200]}...")
    print("-" * 60)
    print()

# 5 | End-to-End Testing
---



Ein gutes RAG-System braucht systematische Tests.  




| Testtyp | Was wird geprüft? | Methode |
|---------|-------------------|--------|
| **Retrieval-Test** | Werden relevante Dokumente gefunden? | Manuell prüfen |
| **Antwortqualität** | Sind die Antworten korrekt? | Expected vs. Actual |
| **Tool-Wahl** | Wählt der Agent das richtige Tool? | Trace analysieren |

In [ ]:
# End-to-End Test: Vollstaendiger Durchlauf
test_szenarien = [
    {
        "frage": "Welche Methode hat Nadia Khoury entwickelt, um Mondstaub zu nutzen, und für welches Projekt arbeitet sie?",
        "erwartetes_tool": "suche_wissensdatenbank",
        "schluesselwoerter": ["Mondstaub", "Lunar Habitat", "Khoury"]
    },
    {
        "frage": "Welche zwei Fachrichtungen verbindet Raoul Mendez in seiner Arbeit, und was hat sein Projekt EarthEars bereits entdeckt?",
        "erwartetes_tool": "suche_wissensdatenbank",
        "schluesselwoerter": ["Mendez", "EarthEars", "Frosch"]
    },
]

print("=== End-to-End Test ===\n")
run_cfg = {"run_name": "E2E-Test", "tags": ["test", "m14"]}

for i, szenario in enumerate(test_szenarien):
    frage = szenario["frage"]
    print(f"Test {i+1}: {frage}")

    ergebnis = rag_agent.invoke(
        {"messages": [{"role": "user", "content": frage}]},
        config=run_cfg
    )

    antwort = ergebnis["messages"][-1].content

    # Schluesselwoerter pruefen
    treffer = sum(1 for kw in szenario["schluesselwoerter"]
                 if kw.lower() in antwort.lower())
    gesamt = len(szenario["schluesselwoerter"])

    print(f"  Antwort: {antwort[:150]}...")
    print(f"  Schluesselwoerter: {treffer}/{gesamt} gefunden")
    print(f"  Status: {'PASS' if treffer >= gesamt // 2 else 'FAIL'}")
    print()

**Zusammenfassung: Wann was?**


| Kriterium | RAG-Chain | RAG-Agent  |
|-----------|----------------|----------------|
| Einfache Q&A | ✅ Ideal | Overhead |
| Gemischte Fragen | Nur RAG | ✅ Flexibel |
| Mehrere Wissensquellen | Komplex | ✅ Multi-Tool |
| Nachvollziehbarkeit | ✅ Deterministisch | Trace noetig |
| Einsteiger | ✅ Empfohlen | Fortgeschritten |



**Empfehlung:** Mit RAG-Chain starten, bei Bedarf zum RAG-Agenten upgraden.
""")

# 6 | RAG-Agent mit LangGraph
---

Kap. 3 nutzte `create_agent()` aus LangChain — kompakt, aber intern eine Black Box.  
LangGraph macht denselben ReAct-Loop **explizit sichtbar** und ermöglicht später Memory, HITL und Multi-Agent ohne Umbau.

**Gleiche Tools, gleiche Fragen — anderer Aufbau:**

| | `create_agent` (Kap. 3) | `create_react_agent` (Kap. 6) |
|---|---|---|
| Import | `langchain.agents` | `langgraph.prebuilt` |
| Loop | Verborgen | Knoten + Kanten explizit |
| Memory / Checkpointing | ❌ | ✅ `MemorySaver` |
| Erweiterbar für M16–M19 | Eingeschränkt | ✅ direkt |

> Die Tools `suche_wissensdatenbank`, `erklaere_konzept` und `berechne_tokens`  
> sowie der Vectorstore aus Kap. 2 werden unverändert wiederverwendet.

In [ ]:
from langgraph.prebuilt import create_react_agent
from langgraph.checkpoint.memory import MemorySaver
import time

# MemorySaver: speichert Gespraechsverlauf innerhalb einer Session
memory = MemorySaver()

# RAG-Agent mit LangGraph — gleiche Tools wie in Kap. 3
rag_agent_lg = create_react_agent(
    model=llm,
    tools=tools,
    prompt=system_prompt,          # system_prompt aus Kap. 3 wiederverwenden
    checkpointer=memory,
)

# Session-ID: eindeutig pro Gespraech (verhindert Ueberlappung bei Testlaeufen)
session = {"configurable": {"thread_id": f"m14-lg-{int(time.time())}"}}

print("RAG-Agent (LangGraph) erstellt")
print(f"Tools: {[t.name for t in tools]}")
print(f"Checkpointer: MemorySaver (session-basiert)")

In [ ]:
from IPython.display import Image as IPImage
display(IPImage(rag_agent_lg.get_graph().draw_mermaid_png()))

In [ ]:
# Vergleich: LangChain vs. LangGraph — gleiche Fragen, beide Agenten
test_fragen_lg = [
    "Was hat Dorian Blackwood vor seiner Karriere bei IllusionSec gemacht, und wie hilft er heute Kindern?",
    "Welche zwei Fachrichtungen verbindet Raoul Mendez in seiner Arbeit, und was hat sein Projekt EarthEars bereits entdeckt?",
]

run_cfg_lg = {"run_name": "RAG-Agent-LangGraph", "tags": ["rag-agent", "langgraph", "m14"]}

for frage in test_fragen_lg:
    mprint(f"**Frage:** {frage}")

    # LangChain-Agent (Kap. 3)
    antwort_lc = rag_agent.invoke(
        {"messages": [{"role": "user", "content": frage}]},
        config=run_cfg_lg
    )["messages"][-1].content

    # LangGraph-Agent (Kap. 6)
    antwort_lg = rag_agent_lg.invoke(
        {"messages": [{"role": "user", "content": frage}]},
        config={**session, **run_cfg_lg}
    )["messages"][-1].content

    print(f"LangChain: {antwort_lc[:180]}...")
    print(f"LangGraph: {antwort_lg[:180]}...")
    print("-" * 60)
    print()

In [ ]:
#@markdown   <p><font size="4" color='green'>  LangSmith Trace-Analyse</font> </br></p>

import time as _t; _t.sleep(2)
show_trace("M14-RAG-Agent", limit=3, show_steps=True)

# A | Aufgabe
---

<p><font color='darkblue' size="4">
📌 <b>Wichtig</b>
</font></p>

Die Aufgabestellungen unten bieten Anregungen, Sie können aber auch gerne eine andere Herausforderung angehen.

**Hinweis zur Lösungshilfe:**
> In diesem Kurs dürfen und sollen Sie generative KI auch als Unterstützung beim Lernen und Entwickeln nutzen. Wenn Sie bei einer Aufgabe festhängen, können Sie zum Beispiel Gemini in Google Colab verwenden, um Fehlermeldungen besser zu verstehen, Ideen für Teilschritte zu bekommen oder Code-Varianten zu prüfen.
> <br>**Wichtig ist nur:** Die KI dient als Lern- und Entwicklungshilfe. Der Schwerpunkt des Kurses bleibt darauf, KI-Agenten selbst zu verstehen, aufzubauen und gezielt weiterzuentwickeln.


**Grundlagen**
- Erweitern Sie den RAG-Agenten aus diesem Modul um eine Biografien-Wissensdatenbank.
- Laden und indexieren Sie die Biografien-Dateien aus `../02_daten/01_text/` in einer neuen Collection.
- Speichern Sie den fertigen Agenten in der Variable `mein_rag_agent`.

**Aufbau**
- Erstellen Sie ein `suche_biografien`-Tool, das diese Collection abfragt.
- Bauen Sie einen Agenten mit beiden Tools: `suche_wissensdatenbank` und `suche_biografien`.
- Testen Sie den Agenten mit 3 Fragen: eine ueber Kurs-Inhalte, eine ueber Biografien, eine gemischte.

**Vertiefung**
- Fuegen Sie einen LangSmith-Test hinzu, der die Tool-Wahl des Agenten protokolliert.
- Pruefen Sie dabei explizit, ob bei den drei Fragetypen das passende Tool gewaehlt wurde.
- Dokumentieren Sie Faelle, in denen der Agent das falsche Tool waehlt, und begruenden Sie moegliche Ursachen.

<p><font color='black' size="5">
🔍 Selbstcheck mit `assert`
</font></p>

`assert` prüft eine Bedingung — ist sie `False`, stoppt Python mit einem `AssertionError` und zeigt die Fehlermeldung an:

```python
assert bedingung, "Fehlermeldung"

assert 2 + 2 == 4, "Rechnung falsch"    # ✅ kein Fehler
assert len("Hi") > 5, "Text zu kurz"   # ❌ AssertionError: Text zu kurz
```

**So nutzen Sie den Selbstcheck:**
1. Erstellen Sie Ihren Biografien-RAG-Agenten in den Zellen über diesem Block
2. Speichern Sie den Agenten in **`mein_rag_agent`**
3. Führen Sie die Zelle unten aus — beide Wissensbereiche (Kurs-Inhalte & Biografien) werden geprüft


<p><font color='black' size="5">
✅ Selbstcheck
</font></p>

In [ ]:
# ► Speichere deinen RAG-Agenten in 'mein_rag_agent'.

_ag = mein_rag_agent  # ← Variablennamen anpassen

assert hasattr(_ag, "invoke"), \
    "❌ Kein invoke() – wurde der Agent mit create_agent() erstellt?"

for _frage, _label in [
    ("Was ist ein Embedding-Vektor?",  "Kurs-Inhalte"),
    ("Wer war Marie Curie?",           "Biografie"),
]:
    _r    = _ag.invoke({"messages": [{"role": "user", "content": _frage}]})
    _msgs = _r["messages"]
    _tool_calls = [m for m in _msgs if getattr(m, "type", None) == "tool"]
    _antwort    = _msgs[-1].content
    assert len(_antwort) > 20, \
        f"❌ Antwort zu kurz für '{_frage}' ({len(_antwort)} Z.) – prüfe Tools und Retriever"
    print(f"✅ {_label} · {len(_tool_calls)} Tool-Aufruf(e) · {_antwort[:80]}...")
